In [1]:
import ollama
import json, numpy as np, pandas as pd

MODEL = "gemma4:31b-cloud"

AI Travel Concierge — a smart assistant that plans complete trips for travellers, end to end. You tell it where you want to go, your dates, and your budget, and it helps you sort out flights, hotels, activities, weather, currency, and even visa rules.

### FIRST STEP IS OF WRITING PROPER SYSTEM INSTRUCTION

In [2]:
### TOO VAGUE PROMPT 
system_vague = "You are a helpful travel assistant. Help the user plan their trip."

In [3]:
### TOO SPECIFIC SYSTEM PROMPT 
system_overspecified = """
You are a travel assistant for Indian travelers. Use the following guidance when helping users.

Popular destinations:
- Goa: recommend Baga Beach, Calangute, and Anjuna. In December, note that prices
  are around 40% higher; during the monsoon (June-Sept), note that many beach shacks are closed.
- Manali: recommend Solang Valley and Rohtang Pass; mention that Rohtang Pass needs a permit.
- Kerala: recommend Alleppey houseboats and Munnar.
- Rajasthan: recommend Jaipur, Udaipur, and Jaisalmer, best visited in winter.

Budget guidance:
- Under 10000 INR: suggest trains and hostels.
- 10000-30000 INR: suggest flights and 3-star hotels.
- Above 30000 INR: suggest 4-star and 5-star hotels.

Traveler types:
- Families with kids: suggest kid-friendly resorts.
- Elderly travelers: avoid strenuous treks.
- Honeymooners: suggest the Maldives or Andaman.
- Vegetarian travelers: mention vegetarian restaurant options.

International trips:
- Thailand: visa on arrival. Dubai: tourist visa required. Singapore: e-visa required.

Seasons:
- Summer: prefer hill stations. Winter: prefer Rajasthan or Goa.
"""


In [4]:
## SYSTEM PROMPT AT RIGHT ATTITUDE.

SYSTEM_PROMPT = """
<role>
You are an AI travel concierge for Indian travelers. You help users plan trips
end to end — flights, stays, activities, budgets, and logistics — and you give
clear, decision-ready, honest recommendations.
</role>

<capabilities>
- Planning itineraries and day-by-day schedules
- Comparing flight and hotel options on price, time, and convenience
- Estimating trip budgets and flagging hidden costs
- Practical logistics: visas, weather, local transport, and what to pack
</capabilities>

<behavior>
- If key details are missing, ask for them first: origin city, destination,
  dates (or rough timing), number of travelers, and rough budget.
- Recommend, don't just list. Give specific choices and say why.
- Be honest about trade-offs (e.g. cheaper fare but a longer layover).
- For anything that needs live data — prices, seat availability, weather,
  current visa rules — say so clearly. Never invent prices, flight numbers,
  or policies.
</behavior>

<output>
- Reply in the traveler's language of planning: warm, concise, and practical.
- Default to INR.
- When you present a plan, structure it clearly — a short summary, then the
  day-by-day or option-by-option details, then rough costs.
</output>
"""

resp = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "I want to bring my pet dog along on a 3-day trip to Coorg. Help me plan it."},
    ],
)

print(resp["message"]["content"])          
print("context tokens sent:", resp["prompt_eval_count"])

Coorg is a fantastic choice for a pet-friendly getaway—plenty of open spaces and fresh air. However, traveling with a dog requires a different kind of planning, especially regarding stays and sightseeing.

To give you a precise, decision-ready itinerary and budget, I need a few more details:

1. **Origin City:** Where are you traveling from? (This helps me suggest the best route and travel time).
2. **Mode of Transport:** Are you driving your own car, or are you looking for a rental/taxi? (Most public transport isn't pet-friendly).
3. **Dog's Profile:** Is your dog a high-energy breed or a senior? Are they social with other dogs?
4. **Budget:** Are you looking for a cozy homestay, a luxury resort, or something budget-friendly?
5. **Dates:** Do you have specific dates? (Coorg weather varies wildly between monsoon and winter).

---

### While you get those details ready, here are 3 "Golden Rules" for a pet-trip to Coorg:

**1. The Stay is Everything**
Unlike hotels, **homestays in Coorg*

In [5]:
# Cities covered: Goa, Manali, Jaipur, Kochi, Port Blair, Bangkok
# Origins with flights: Delhi (DEL), Mumbai (BOM)

FLIGHTS = {
    ("DEL", "GOI"): [
        {"flight_no": "6E-231",  "airline": "IndiGo",    "depart": "06:10", "arrive": "08:45", "duration_min": 155, "stops": 0, "price_inr": 5499, "seats_left": 7},
        {"flight_no": "6E-577",  "airline": "IndiGo",    "depart": "11:20", "arrive": "13:55", "duration_min": 155, "stops": 0, "price_inr": 5899, "seats_left": 12},
        {"flight_no": "AI-573",  "airline": "Air India", "depart": "08:30", "arrive": "11:05", "duration_min": 155, "stops": 0, "price_inr": 6120, "seats_left": 3},
        {"flight_no": "UK-849",  "airline": "Vistara",   "depart": "09:45", "arrive": "12:20", "duration_min": 155, "stops": 0, "price_inr": 7250, "seats_left": 5},
        {"flight_no": "SG-133",  "airline": "SpiceJet",  "depart": "14:05", "arrive": "16:40", "duration_min": 155, "stops": 0, "price_inr": 5299, "seats_left": 9},
        {"flight_no": "QP-1401", "airline": "Akasa Air", "depart": "18:30", "arrive": "21:05", "duration_min": 155, "stops": 0, "price_inr": 5650, "seats_left": 4},
        {"flight_no": "6E-6133", "airline": "IndiGo",    "depart": "07:00", "arrive": "11:40", "duration_min": 280, "stops": 1, "price_inr": 4899, "seats_left": 6},
        {"flight_no": "AI-665",  "airline": "Air India", "depart": "12:15", "arrive": "17:20", "duration_min": 305, "stops": 1, "price_inr": 5100, "seats_left": 8},
    ],
    ("DEL", "JAI"): [
        {"flight_no": "6E-2011", "airline": "IndiGo",    "depart": "07:30", "arrive": "08:25", "duration_min": 55, "stops": 0, "price_inr": 2899, "seats_left": 14},
        {"flight_no": "UK-621",  "airline": "Vistara",   "depart": "10:15", "arrive": "11:10", "duration_min": 55, "stops": 0, "price_inr": 3450, "seats_left": 6},
        {"flight_no": "AI-9803", "airline": "Air India", "depart": "16:40", "arrive": "17:35", "duration_min": 55, "stops": 0, "price_inr": 3100, "seats_left": 9},
        {"flight_no": "SG-2461", "airline": "SpiceJet",  "depart": "20:05", "arrive": "21:00", "duration_min": 55, "stops": 0, "price_inr": 2750, "seats_left": 11},
    ],
    ("DEL", "COK"): [
        {"flight_no": "6E-509",  "airline": "IndiGo",    "depart": "05:50", "arrive": "09:05", "duration_min": 195, "stops": 0, "price_inr": 7899, "seats_left": 5},
        {"flight_no": "AI-467",  "airline": "Air India", "depart": "09:20", "arrive": "12:40", "duration_min": 200, "stops": 0, "price_inr": 8250, "seats_left": 4},
        {"flight_no": "UK-833",  "airline": "Vistara",   "depart": "13:10", "arrive": "16:30", "duration_min": 200, "stops": 0, "price_inr": 9100, "seats_left": 7},
        {"flight_no": "SG-711",  "airline": "SpiceJet",  "depart": "18:45", "arrive": "23:35", "duration_min": 290, "stops": 1, "price_inr": 6990, "seats_left": 10},
    ],
    ("DEL", "BKK"): [
        {"flight_no": "6E-1051", "airline": "IndiGo",       "depart": "23:15", "arrive": "05:10", "duration_min": 265, "stops": 0, "price_inr": 14500, "seats_left": 8},
        {"flight_no": "TG-316",  "airline": "Thai Airways", "depart": "08:30", "arrive": "14:15", "duration_min": 255, "stops": 0, "price_inr": 21800, "seats_left": 3},
        {"flight_no": "AI-332",  "airline": "Air India",    "depart": "13:05", "arrive": "18:55", "duration_min": 260, "stops": 0, "price_inr": 18900, "seats_left": 6},
        {"flight_no": "SG-87",   "airline": "SpiceJet",     "depart": "21:40", "arrive": "03:30", "duration_min": 260, "stops": 0, "price_inr": 12750, "seats_left": 12},
    ],
    ("BOM", "GOI"): [
        {"flight_no": "6E-441",  "airline": "IndiGo",   "depart": "07:20", "arrive": "08:30", "duration_min": 70, "stops": 0, "price_inr": 3299, "seats_left": 15},
        {"flight_no": "UK-771",  "airline": "Vistara",  "depart": "12:10", "arrive": "13:20", "duration_min": 70, "stops": 0, "price_inr": 4100, "seats_left": 8},
        {"flight_no": "QP-1503", "airline": "Akasa Air","depart": "17:45", "arrive": "18:55", "duration_min": 70, "stops": 0, "price_inr": 3550, "seats_left": 6},
    ],
}

HOTELS = {
    "Goa": [
        {"hotel_id": "GOA-101", "name": "Beira Mar Resort",    "area": "Baga",      "stars": 4, "price_per_night_inr": 5200, "rating": 4.3, "breakfast": True},
        {"hotel_id": "GOA-102", "name": "Sunset Sands",        "area": "Calangute", "stars": 3, "price_per_night_inr": 3100, "rating": 4.0, "breakfast": True},
        {"hotel_id": "GOA-103", "name": "Palm Grove Boutique", "area": "Anjuna",    "stars": 4, "price_per_night_inr": 6100, "rating": 4.5, "breakfast": True},
        {"hotel_id": "GOA-104", "name": "Backpacker's Nest",   "area": "Vagator",   "stars": 2, "price_per_night_inr": 1400, "rating": 3.8, "breakfast": False},
        {"hotel_id": "GOA-105", "name": "Coco Beach Villas",   "area": "Candolim",  "stars": 5, "price_per_night_inr": 9800, "rating": 4.7, "breakfast": True},
        {"hotel_id": "GOA-106", "name": "Riverside Retreat",   "area": "Panaji",    "stars": 3, "price_per_night_inr": 2800, "rating": 4.1, "breakfast": True},
    ],
    "Manali": [
        {"hotel_id": "MAN-201", "name": "Snow Valley Resort",  "area": "Log Huts Area", "stars": 4, "price_per_night_inr": 4200, "rating": 4.2, "breakfast": True},
        {"hotel_id": "MAN-202", "name": "Riverbank Cottages",  "area": "Old Manali",    "stars": 3, "price_per_night_inr": 2500, "rating": 4.4, "breakfast": True},
        {"hotel_id": "MAN-203", "name": "The Himalayan",       "area": "Hadimba Road",  "stars": 5, "price_per_night_inr": 11000,"rating": 4.6, "breakfast": True},
        {"hotel_id": "MAN-204", "name": "Backpacker Panda",    "area": "Vashisht",      "stars": 2, "price_per_night_inr": 1200, "rating": 3.9, "breakfast": False},
    ],
    "Jaipur": [
        {"hotel_id": "JAI-301", "name": "Pink City Haveli",    "area": "C-Scheme",   "stars": 4, "price_per_night_inr": 4800, "rating": 4.5, "breakfast": True},
        {"hotel_id": "JAI-302", "name": "Rambagh View",        "area": "Bani Park",  "stars": 3, "price_per_night_inr": 2600, "rating": 4.1, "breakfast": True},
        {"hotel_id": "JAI-303", "name": "Royal Fort Palace",   "area": "Amer Road",  "stars": 5, "price_per_night_inr": 13500,"rating": 4.8, "breakfast": True},
        {"hotel_id": "JAI-304", "name": "Zostel Jaipur",       "area": "Hathroi",    "stars": 2, "price_per_night_inr": 1100, "rating": 4.0, "breakfast": False},
    ],
    "Kochi": [
        {"hotel_id": "COK-401", "name": "Backwater Breeze",    "area": "Fort Kochi", "stars": 4, "price_per_night_inr": 5000, "rating": 4.4, "breakfast": True},
        {"hotel_id": "COK-402", "name": "Marine Drive Inn",    "area": "Ernakulam",  "stars": 3, "price_per_night_inr": 2900, "rating": 4.0, "breakfast": True},
        {"hotel_id": "COK-403", "name": "Spice Coast Villas",  "area": "Fort Kochi", "stars": 5, "price_per_night_inr": 9500, "rating": 4.7, "breakfast": True},
        {"hotel_id": "COK-404", "name": "Tea Trail Hostel",    "area": "MG Road",    "stars": 2, "price_per_night_inr": 1300, "rating": 3.8, "breakfast": False},
    ],
    "Port Blair": [
        {"hotel_id": "IXZ-501", "name": "Coral Reef Resort",   "area": "Marine Hill",    "stars": 4, "price_per_night_inr": 6200, "rating": 4.3, "breakfast": True},
        {"hotel_id": "IXZ-502", "name": "Island View Inn",     "area": "Aberdeen",       "stars": 3, "price_per_night_inr": 3400, "rating": 4.0, "breakfast": True},
        {"hotel_id": "IXZ-503", "name": "Barefoot Bay",        "area": "Corbyn's Cove",  "stars": 5, "price_per_night_inr": 12500,"rating": 4.8, "breakfast": True},
        {"hotel_id": "IXZ-504", "name": "Divers Lodge",        "area": "Phoenix Bay",    "stars": 2, "price_per_night_inr": 1600, "rating": 3.9, "breakfast": False},
    ],
    "Bangkok": [
        {"hotel_id": "BKK-601", "name": "Riverside Grand",     "area": "Silom",     "stars": 4, "price_per_night_inr": 5500, "rating": 4.4, "breakfast": True},
        {"hotel_id": "BKK-602", "name": "Sukhumvit Stay",      "area": "Sukhumvit", "stars": 3, "price_per_night_inr": 3200, "rating": 4.1, "breakfast": True},
        {"hotel_id": "BKK-603", "name": "Chao Phraya Luxury",  "area": "Riverside", "stars": 5, "price_per_night_inr": 10500,"rating": 4.7, "breakfast": True},
        {"hotel_id": "BKK-604", "name": "Khao San Hostel",     "area": "Khao San",  "stars": 2, "price_per_night_inr": 900,  "rating": 4.0, "breakfast": False},
    ],
}

# Each city has a "typical" climate plus a few specific months.
WEATHER = {
    "Goa":        {"typical": {"high_c": 31, "low_c": 24, "rain_mm": 40,  "summary": "Warm, humid; dry Nov-Feb"},
                   12: {"high_c": 32, "low_c": 21, "rain_mm": 18,  "summary": "Warm, dry, sunny — peak season"}},
    "Manali":     {"typical": {"high_c": 15, "low_c": 5,  "rain_mm": 60,  "summary": "Cool hills; snow in winter"},
                   12: {"high_c": 8,  "low_c": -2, "rain_mm": 30,  "summary": "Cold, chance of snowfall"}},
    "Jaipur":     {"typical": {"high_c": 34, "low_c": 22, "rain_mm": 30,  "summary": "Hot summers, pleasant winters"},
                   12: {"high_c": 24, "low_c": 9,  "rain_mm": 5,   "summary": "Cool, dry — ideal for sightseeing"}},
    "Kochi":      {"typical": {"high_c": 31, "low_c": 25, "rain_mm": 120, "summary": "Tropical, humid, heavy monsoon"},
                   12: {"high_c": 31, "low_c": 24, "rain_mm": 30,  "summary": "Warm, drier — good time to visit"}},
    "Port Blair": {"typical": {"high_c": 30, "low_c": 24, "rain_mm": 150, "summary": "Tropical islands, humid"},
                   12: {"high_c": 29, "low_c": 23, "rain_mm": 60,  "summary": "Pleasant, calmer seas"}},
    "Bangkok":    {"typical": {"high_c": 34, "low_c": 26, "rain_mm": 100, "summary": "Hot and humid year-round"},
                   12: {"high_c": 31, "low_c": 22, "rain_mm": 10,  "summary": "Cooler, dry — best season"}},
}

CURRENCY = {"INR": 1.0, "USD": 83.2, "EUR": 90.1, "GBP": 105.7, "AED": 22.6, "THB": 2.3, "SGD": 61.4}

VISA = {
    ("Indian", "Thailand"):  {"required": True,  "type": "Visa on arrival",      "stay_days": 15, "fee_inr": 2200},
    ("Indian", "UAE"):       {"required": True,  "type": "e-Visa (pre-arrival)", "stay_days": 30, "fee_inr": 6500},
    ("Indian", "Singapore"): {"required": True,  "type": "e-Visa (pre-arrival)", "stay_days": 30, "fee_inr": 2700},
    ("Indian", "Indonesia"): {"required": True,  "type": "Visa on arrival",      "stay_days": 30, "fee_inr": 3000},
    ("Indian", "Maldives"):  {"required": True,  "type": "Free visa on arrival", "stay_days": 30, "fee_inr": 0},
    ("Indian", "Nepal"):     {"required": False, "type": "No visa required",     "stay_days": 150,"fee_inr": 0},
}

ACTIVITIES = {
    "Goa": [
        {"name": "Dudhsagar Falls day trip",  "category": "nature",    "area": "Mollem",  "price_inr": 1500},
        {"name": "Scuba diving at Grande Is.","category": "adventure", "area": "Grande",  "price_inr": 3500},
        {"name": "Old Goa churches walk",     "category": "heritage",  "area": "Old Goa", "price_inr": 0},
        {"name": "Sunset cruise on the Mandovi","category": "leisure", "area": "Panaji",  "price_inr": 700},
    ],
    "Manali": [
        {"name": "Solang Valley ropeway",     "category": "adventure", "area": "Solang",   "price_inr": 800},
        {"name": "Rohtang Pass day trip",     "category": "nature",    "area": "Rohtang",  "price_inr": 2500},
        {"name": "Hadimba Temple visit",      "category": "heritage",  "area": "Manali",   "price_inr": 0},
        {"name": "Paragliding at Solang",     "category": "adventure", "area": "Solang",   "price_inr": 1800},
    ],
    "Jaipur": [
        {"name": "Amber Fort tour",           "category": "heritage",  "area": "Amer",     "price_inr": 550},
        {"name": "City Palace & Jantar Mantar","category": "heritage", "area": "Old City", "price_inr": 700},
        {"name": "Nahargarh Fort sunset",     "category": "leisure",   "area": "Nahargarh","price_inr": 200},
        {"name": "Chokhi Dhani cultural eve", "category": "culture",   "area": "Tonk Road","price_inr": 1100},
    ],
    "Kochi": [
        {"name": "Backwater houseboat cruise","category": "nature",    "area": "Alleppey", "price_inr": 6000},
        {"name": "Chinese fishing nets walk", "category": "heritage",  "area": "Fort Kochi","price_inr": 0},
        {"name": "Kathakali dance show",      "category": "culture",   "area": "Fort Kochi","price_inr": 500},
        {"name": "Mattancherry Palace",       "category": "heritage",  "area": "Mattancherry","price_inr": 100},
    ],
    "Port Blair": [
        {"name": "Cellular Jail light & sound","category": "heritage", "area": "Port Blair","price_inr": 300},
        {"name": "Radhanagar Beach (Havelock)","category": "leisure",  "area": "Havelock", "price_inr": 0},
        {"name": "Scuba diving at Havelock",  "category": "adventure", "area": "Havelock", "price_inr": 4000},
        {"name": "Ross Island half-day",      "category": "nature",    "area": "Ross Is.", "price_inr": 1200},
    ],
    "Bangkok": [
        {"name": "Grand Palace & Wat Phra Kaew","category": "heritage","area": "Rattanakosin","price_inr": 1200},
        {"name": "Chatuchak weekend market",  "category": "shopping",  "area": "Chatuchak","price_inr": 0},
        {"name": "Wat Arun river visit",       "category": "heritage", "area": "Thonburi", "price_inr": 250},
        {"name": "Damnoen floating market",   "category": "culture",   "area": "Ratchaburi","price_inr": 1500},
    ],
}

TRAINS = {
    ("DEL", "GOI"): [
        {"train_no": "12450", "name": "Goa Sampark Kranti",    "depart": "10:55", "arrive": "05:30 (+2d)", "duration_h": 42, "fare_inr": 2150, "class": "3A"},
        {"train_no": "22414", "name": "Nizamuddin-Madgaon SF", "depart": "11:25", "arrive": "20:40 (+1d)", "duration_h": 33, "fare_inr": 2600, "class": "3A"},
    ],
    ("DEL", "JAI"): [
        {"train_no": "12015", "name": "Ajmer Shatabdi",        "depart": "06:05", "arrive": "10:30",       "duration_h": 4.5,"fare_inr": 900,  "class": "CC"},
        {"train_no": "12958", "name": "ADI SJ Rajdhani",       "depart": "19:55", "arrive": "00:05 (+1d)", "duration_h": 4.2,"fare_inr": 1450, "class": "3A"},
    ],
}

RESTAURANTS = {
    "Goa":        [{"name": "Gunpowder", "cuisine": "South Indian", "veg": True, "area": "Assagao", "avg_cost_inr": 900},
                   {"name": "Fisherman's Wharf", "cuisine": "Goan", "veg": False, "area": "Panaji", "avg_cost_inr": 1200}],
    "Manali":     [{"name": "Cafe 1947", "cuisine": "Italian", "veg": True, "area": "Old Manali", "avg_cost_inr": 700},
                   {"name": "Johnson's Cafe", "cuisine": "Continental", "veg": False, "area": "Circuit House Rd", "avg_cost_inr": 1100}],
    "Jaipur":     [{"name": "LMB", "cuisine": "Rajasthani", "veg": True, "area": "Johari Bazaar", "avg_cost_inr": 600},
                   {"name": "Handi", "cuisine": "Mughlai", "veg": False, "area": "MI Road", "avg_cost_inr": 900}],
    "Kochi":      [{"name": "Kashi Art Cafe", "cuisine": "Cafe", "veg": True, "area": "Fort Kochi", "avg_cost_inr": 700},
                   {"name": "Fort House", "cuisine": "Seafood", "veg": False, "area": "Fort Kochi", "avg_cost_inr": 1100}],
    "Port Blair": [{"name": "Annapurna", "cuisine": "South Indian", "veg": True, "area": "Aberdeen", "avg_cost_inr": 400},
                   {"name": "New Lighthouse", "cuisine": "Seafood", "veg": False, "area": "Marine Hill", "avg_cost_inr": 1000}],
    "Bangkok":    [{"name": "May Veggie Home", "cuisine": "Vegan", "veg": True, "area": "Sukhumvit", "avg_cost_inr": 800},
                   {"name": "Jay Fai", "cuisine": "Thai", "veg": False, "area": "Old Town", "avg_cost_inr": 2500}],
}

TRANSPORT = {
    "Goa":        [{"mode": "Scooter rental", "note": "Cheapest way around", "approx_cost_inr": 400},
                   {"mode": "App cab (Goa Miles)", "note": "Reliable for airport", "approx_cost_inr": 1000}],
    "Manali":     [{"mode": "Local taxi", "note": "Fixed-rate sightseeing", "approx_cost_inr": 2000},
                   {"mode": "HRTC bus", "note": "Budget intercity", "approx_cost_inr": 150}],
    "Jaipur":     [{"mode": "Auto-rickshaw", "note": "Short city hops", "approx_cost_inr": 150},
                   {"mode": "App cab (Ola/Uber)", "note": "AC, metered", "approx_cost_inr": 400}],
    "Kochi":      [{"mode": "Ferry", "note": "Scenic, connects islands", "approx_cost_inr": 40},
                   {"mode": "App cab", "note": "For Ernakulam-Fort Kochi", "approx_cost_inr": 350}],
    "Port Blair": [{"mode": "Auto-rickshaw", "note": "Within Port Blair", "approx_cost_inr": 200},
                   {"mode": "Govt ferry to Havelock", "note": "Book early", "approx_cost_inr": 600}],
    "Bangkok":    [{"mode": "BTS Skytrain", "note": "Fast, avoids traffic", "approx_cost_inr": 120},
                   {"mode": "Tuk-tuk", "note": "Short hops, negotiate fare", "approx_cost_inr": 250}],
}

ADVISORY = {
    "India":     {"level": "Normal", "note": "No major concerns for domestic travel."},
    "Thailand":  {"level": "Exercise caution", "note": "Watch for monsoon flooding in the south (Sep-Nov)."},
    "Indonesia": {"level": "Exercise caution", "note": "Volcanic activity possible in parts of Bali/Java."},
    "Maldives":  {"level": "Normal", "note": "Standard precautions; respect local laws."},
}

HOLIDAYS = {
    ("India", 12): [{"date": "2026-12-25", "name": "Christmas"}, {"date": "2026-12-31", "name": "New Year's Eve"}],
    ("India", 1):  [{"date": "2027-01-26", "name": "Republic Day"}],
}

AIRPORTS = {
    "Delhi":      {"code": "DEL", "name": "Indira Gandhi International", "terminals": 3},
    "Mumbai":     {"code": "BOM", "name": "Chhatrapati Shivaji Maharaj", "terminals": 2},
    "Goa":        {"code": "GOI", "name": "Dabolim / Manohar (Mopa)",   "terminals": 2},
    "Jaipur":     {"code": "JAI", "name": "Jaipur International",        "terminals": 1},
    "Kochi":      {"code": "COK", "name": "Cochin International",        "terminals": 3},
    "Port Blair": {"code": "IXZ", "name": "Veer Savarkar International", "terminals": 1},
    "Bangkok":    {"code": "BKK", "name": "Suvarnabhumi",               "terminals": 1},
}

In [6]:
### TOOLS 
def search_flights(origin_code: str, destination_code: str, date: str) -> list:
    """Search available flights between two airports on a date.

    Args:
        origin_code: departure airport IATA code, e.g. "DEL".
        destination_code: arrival airport IATA code, e.g. "GOI".
        date: travel date as YYYY-MM-DD.
    Returns:
        List of flights, each with: flight_no, airline, depart, arrive,
        duration_min, stops, price_inr, seats_left.
    """
    return FLIGHTS.get((origin_code, destination_code), [])


def search_hotels(city: str, max_price_inr: int = 100000, min_stars: int = 0) -> list:
    """Find hotels in a city within a price and star-rating filter.

    Args:
        city: city name, e.g. "Goa".
        max_price_inr: maximum price per night in INR.
        min_stars: minimum star rating (0-5).
    Returns:
        List of hotels, each with: hotel_id, name, area, stars,
        price_per_night_inr, rating, breakfast.
    """
    return [h for h in HOTELS.get(city, [])
            if h["price_per_night_inr"] <= max_price_inr and h["stars"] >= min_stars]


def get_weather(city: str, month: int) -> dict:
    """Get typical weather for a city in a given month.

    Args:
        city: city name.
        month: month number 1-12.
    Returns:
        Dict with high_c, low_c, rain_mm, summary. Falls back to the city's
        typical climate if that month isn't on file.
    """
    city_data = WEATHER.get(city, {})
    return city_data.get(month, city_data.get("typical", {}))


def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount from one currency to another.

    Args:
        amount: amount to convert.
        from_currency: source currency code, e.g. "USD".
        to_currency: target currency code, e.g. "INR".
    Returns:
        Dict with converted_amount and rate.
    """
    inr = amount * CURRENCY[from_currency]
    converted = inr / CURRENCY[to_currency]
    return {"converted_amount": round(converted, 2),
            "rate": round(CURRENCY[from_currency] / CURRENCY[to_currency], 4)}


def get_visa_rules(nationality: str, destination_country: str) -> dict:
    """Get visa requirements for a traveler's nationality and destination.

    Args:
        nationality: traveler nationality, e.g. "Indian".
        destination_country: destination country, e.g. "Thailand".
    Returns:
        Dict with required, type, stay_days, fee_inr. Falls back to a
        'verify with embassy' note if the pair is unknown.
    """
    return VISA.get((nationality, destination_country),
                    {"required": None, "note": "Not on file — verify with the country's embassy."})


def get_flight_status(flight_no: str) -> dict:
    """Get the live status of a flight by its number.

    Args:
        flight_no: flight number, e.g. "6E-231".
    Returns:
        Dict with flight_no, status, and gate.
    """
    return {"flight_no": flight_no, "status": "On time", "gate": "12B"}


def search_activities(city: str, category: str = "") -> list:
    """Find things to do in a city, optionally filtered by category.

    Args:
        city: city name.
        category: optional filter, e.g. "adventure", "heritage", "nature".
    Returns:
        List of activities, each with: name, category, area, price_inr.
    """
    items = ACTIVITIES.get(city, [])
    return [a for a in items if not category or a["category"] == category]


def search_trains(origin_code: str, destination_code: str, date: str) -> list:
    """Search trains between two stations on a date.

    Args:
        origin_code: origin station code.
        destination_code: destination station code.
        date: travel date as YYYY-MM-DD.
    Returns:
        List of trains, each with: train_no, name, depart, arrive,
        duration_h, fare_inr, class.
    """
    return TRAINS.get((origin_code, destination_code), [])


def search_restaurants(city: str, cuisine: str = "", veg_only: bool = False) -> list:
    """Find restaurants in a city.

    Args:
        city: city name.
        cuisine: optional cuisine filter, e.g. "Goan".
        veg_only: if True, return only vegetarian restaurants.
    Returns:
        List of restaurants, each with: name, cuisine, veg, area, avg_cost_inr.
    """
    items = RESTAURANTS.get(city, [])
    return [r for r in items
            if (not cuisine or r["cuisine"] == cuisine) and (not veg_only or r["veg"])]


def estimate_trip_budget(nights: int, flight_price_inr: int,
                         hotel_price_per_night_inr: int, travelers: int,
                         daily_spend_inr: int = 2000) -> dict:
    """Estimate a total trip budget.

    Args:
        nights: number of nights.
        flight_price_inr: one-way flight price per person.
        hotel_price_per_night_inr: hotel price per night (per room).
        travelers: number of travelers.
        daily_spend_inr: per-person daily spend on food/local travel.
    Returns:
        Dict with flights, hotel, daily, and total (all INR).
    """
    flights = flight_price_inr * 2 * travelers
    hotel = hotel_price_per_night_inr * nights
    daily = daily_spend_inr * travelers * (nights + 1)
    return {"flights": flights, "hotel": hotel, "daily": daily,
            "total": flights + hotel + daily}


def get_local_transport(city: str) -> list:
    """Get local transport options in a city.

    Args:
        city: city name.
    Returns:
        List of options, each with: mode, note, approx_cost_inr.
    """
    return TRANSPORT.get(city, [])


def get_travel_advisory(country: str) -> dict:
    """Get the current safety advisory for a country.

    Args:
        country: country name.
    Returns:
        Dict with level and note.
    """
    return ADVISORY.get(country, {"level": "Unknown", "note": "No advisory on file."})


def get_packing_list(destination: str, month: int, days: int) -> list:
    """Suggest a packing list based on destination, month, and trip length.

    Args:
        destination: city name.
        month: month number 1-12.
        days: number of days.
    Returns:
        List of items to pack (strings).
    """
    base = ["ID card", "phone charger", f"{days + 1} sets of clothes", "toiletries"]
    w = WEATHER.get(destination, {}).get(month, WEATHER.get(destination, {}).get("typical", {}))
    if w and w["high_c"] >= 30:
        base += ["sunscreen", "sunglasses", "light cotton wear"]
    if w and w["high_c"] <= 10:
        base += ["heavy jacket", "woollen cap", "gloves"]
    if w and w["rain_mm"] > 80:
        base += ["umbrella", "rain jacket"]
    return base


def get_public_holidays(country: str, month: int) -> list:
    """Get public holidays in a country for a given month (affects crowds/prices).

    Args:
        country: country name.
        month: month number 1-12.
    Returns:
        List of holidays, each with: date, name.
    """
    return HOLIDAYS.get((country, month), [])


def get_airport_info(city: str) -> dict:
    """Get airport details for a city.

    Args:
        city: city name.
    Returns:
        Dict with code, name, terminals.
    """
    return AIRPORTS.get(city, {})


def book_flight(flight_no: str, passenger_name: str, travel_date: str) -> dict:
    """Book a flight seat (demo — returns a mock confirmation).

    Args:
        flight_no: flight number to book.
        passenger_name: full name of the passenger.
        travel_date: travel date as YYYY-MM-DD.
    Returns:
        Dict with pnr, flight_no, passenger, date, status.
    """
    return {"pnr": "PNR" + str(abs(hash(flight_no + passenger_name)) % 1000000),
            "flight_no": flight_no, "passenger": passenger_name,
            "date": travel_date, "status": "CONFIRMED"}


REAL_TOOLS = [
    search_flights, search_hotels, get_weather, convert_currency, get_visa_rules,
    get_flight_status, search_activities, search_trains, search_restaurants,
    estimate_trip_budget, get_local_transport, get_travel_advisory,
    get_packing_list, get_public_holidays, get_airport_info, book_flight,
]
REGISTRY = {f.__name__: f for f in REAL_TOOLS}

In [7]:
REGISTRY

{'search_flights': <function __main__.search_flights(origin_code: str, destination_code: str, date: str) -> list>,
 'search_hotels': <function __main__.search_hotels(city: str, max_price_inr: int = 100000, min_stars: int = 0) -> list>,
 'get_weather': <function __main__.get_weather(city: str, month: int) -> dict>,
 'convert_currency': <function __main__.convert_currency(amount: float, from_currency: str, to_currency: str) -> dict>,
 'get_visa_rules': <function __main__.get_visa_rules(nationality: str, destination_country: str) -> dict>,
 'get_flight_status': <function __main__.get_flight_status(flight_no: str) -> dict>,
 'search_activities': <function __main__.search_activities(city: str, category: str = '') -> list>,
 'search_trains': <function __main__.search_trains(origin_code: str, destination_code: str, date: str) -> list>,
 'search_restaurants': <function __main__.search_restaurants(city: str, cuisine: str = '', veg_only: bool = False) -> list>,
 'estimate_trip_budget': <function

In [8]:
## CALLING TOOL
resp = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "hi"},
    ],
    tools=REAL_TOOLS,
)

print(resp.message.content)
print("tool calls:", resp.message.tool_calls)      # None — "hi" needs no tool
print("context tokens sent:", resp.prompt_eval_count)

Hello! I'm your AI travel concierge. I'm here to help you plan your perfect trip, from the first flight search to the final packing list.

Whether you're dreaming of a relaxing beach getaway, a mountain adventure, or a quick city break, I've got you covered with flights, hotels, itineraries, and budget estimates.

To get started, tell me a little bit about what you're looking for:
1. **Where do you want to go?** (Or should I suggest some destinations?)
2. **Where are you flying/traveling from?**
3. **When are you planning to travel and for how long?**
4. **Who is coming along?** (Solo, partner, family, or friends?)
5. **What is your rough budget?** (Budget-friendly, mid-range, or luxury?)

Looking forward to planning your next adventure!
tool calls: None
context tokens sent: 1944


In [9]:
## FOR EFFECTIVE TOOL CALLING FIRST THING WE NEED TO DO IS WRITE PROPER TOOLS AS PROPER TOOLS WILL CONVERT TO BETTER SCHEMA..

In [10]:
from ollama._utils import convert_function_to_tool
import json

schema = convert_function_to_tool(search_flights).model_dump(exclude_none=True)
print(json.dumps(schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "search_flights",
    "description": "Search available flights between two airports on a date.",
    "parameters": {
      "type": "object",
      "required": [
        "origin_code",
        "destination_code",
        "date"
      ],
      "properties": {
        "origin_code": {
          "type": "string",
          "description": "departure airport IATA code, e.g. \"DEL\"."
        },
        "destination_code": {
          "type": "string",
          "description": "arrival airport IATA code, e.g. \"GOI\"."
        },
        "date": {
          "type": "string",
          "description": "travel date as YYYY-MM-DD."
        }
      }
    }
  }
}


In [11]:
###  DEFERRED LOADING

In [12]:
# This function is a very simple tool search function. Its job is to look through all available tools and return the names of the tools 
# that seem relevant to the user's query.


def search_tools(query, max_results=3):
    words = query.lower().split()
    hits = []
    for f in REAL_TOOLS:
        text = (f.__name__ + " " + (f.__doc__ or "")).lower()  # checking for word in function name or docstring
        if any(w in text for w in words):
            hits.append(f.__name__)
    return hits[:max_results]

In [13]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_tools",
        "description": (
            "Search for tools that are not yet loaded into this conversation. "
            "Right now, this search tool is the only capability you have - "
            "every other tool must be found through this one before it can "
            "be called. Use it the moment you realize you need a capability "
            "you don't currently have.\n\n"
            "Returns a list of matching tool names. Once a tool is returned "
            "here, it becomes directly callable on your next turn - you do "
            "not need to search for it again later in this conversation.\n\n"
            "If nothing matches, try again with broader or different keywords "
            "rather than giving up."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Short keywords describing the capability you need - "
                        "not a full sentence. "
                        "Good: 'check inventory stock', 'process refund', "
                        "'sales report'. "
                        "Avoid: a complete question or a full description "
                        "of the task."
                    ),
                },
                "max_results": {
                    "type": "integer",
                    "description": (
                        "Maximum number of matching tools to return. "
                        "Defaults to 3. Increase only if the task clearly "
                        "needs several related tools at once."
                    ),
                },
            },
            "required": ["query"],
        },
    },
}

In [14]:
SYSTEM_PROMPT = SYSTEM_PROMPT + """
<Using tools>
- You do NOT keep real travel data in your memory. Real flights, hotels, weather,
  prices, visa rules, trains, restaurants, activities, and budgets must ALWAYS
  come from tools — never from your own guesses.
- Most tools are not loaded yet. You have a `search_tools` tool to find them.
  When a request needs real data: first call `search_tools` with short keywords,
  then call the tool(s) it returns, then answer using only their results.
- Never invent flight numbers, hotel names, prices, or policies.
</Using tools>
"""


In [15]:
print(SYSTEM_PROMPT)


<role>
You are an AI travel concierge for Indian travelers. You help users plan trips
end to end — flights, stays, activities, budgets, and logistics — and you give
clear, decision-ready, honest recommendations.
</role>

<capabilities>
- Planning itineraries and day-by-day schedules
- Comparing flight and hotel options on price, time, and convenience
- Estimating trip budgets and flagging hidden costs
- Practical logistics: visas, weather, local transport, and what to pack
</capabilities>

<behavior>
- If key details are missing, ask for them first: origin city, destination,
  dates (or rough timing), number of travelers, and rough budget.
- Recommend, don't just list. Give specific choices and say why.
- Be honest about trade-offs (e.g. cheaper fare but a longer layover).
- For anything that needs live data — prices, seat availability, weather,
  current visa rules — say so clearly. Never invent prices, flight numbers,
  or policies.
</behavior>

<output>
- Reply in the traveler's la

In [16]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Hi"},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=[SEARCH_TOOL]
        )

In [17]:
print(response)
print("*********"*10)
print(response.message.content)
print("*********"*10)
print(response.prompt_eval_count)

model='gemma4:31b' created_at='2026-09-22T04:43:13.388099367Z' done=True done_reason='stop' total_duration=927544984 load_duration=None prompt_eval_count=690 prompt_eval_duration=None eval_count=210 eval_duration=None message=Message(role='assistant', content="Namaste! I'm your AI travel concierge. I'm here to help you plan your next trip from start to finish—whether it's a relaxing beach getaway, a mountain trek, or an international adventure.\n\nTo give you a decision-ready plan with accurate recommendations, I'll need a few details first:\n\n1. **Where are you heading?** (Or do you need help picking a destination?)\n2. **Where are you flying/traveling from?** (Your origin city in India)\n3. **When are you planning to go and for how long?** (Dates or a rough month/duration)\n4. **Who is traveling?** (Solo, couple, family with kids, or a group of friends)\n5. **What is your rough budget?** (Budget, Mid-range, or Luxury)\n\nOnce you share these, I can get started on flights, stays, a d

In [18]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What's the weather like in Manali in December?"},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=[SEARCH_TOOL]
        )

In [19]:
print(response)

model='gemma4:31b' created_at='2026-09-22T04:43:13.992609833Z' done=True done_reason='stop' total_duration=254362259 load_duration=None prompt_eval_count=701 prompt_eval_duration=None eval_count=18 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_tools', arguments={'query': 'weather Manali December'}))]) logprobs=None


In [20]:
o1= search_tools("weather Manali December")

In [21]:
o1

['get_weather']

In [22]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "I'm flying from Delhi (DEL) to Goa (GOI) on 2026-12-15. "
                            "Find me flights, a 4-star hotel in Goa, and a few things to do there."},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=[SEARCH_TOOL]
        )

In [23]:
print(response)

model='gemma4:31b' created_at='2026-09-22T04:43:14.653406911Z' done=True done_reason='stop' total_duration=267014748 load_duration=None prompt_eval_count=738 prompt_eval_duration=None eval_count=18 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_tools', arguments={'query': 'flights hotel activities Goa'}))]) logprobs=None


In [24]:
o2= search_tools("flights hotels activities")

In [25]:
o2

['search_flights', 'search_hotels', 'search_activities']

### AGENTIC LOOP WITH NORMAL TOOL CALLING

In [26]:
# 1. Initialize the messages list outside the function with just the system prompt
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def run(question, max_turns=10):
    # 2. Append the new user question to the existing external list
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=REAL_TOOLS,
            options={"num_ctx": 8000}
        )

        # the size of the ENTIRE context the model just read
        size = response.prompt_eval_count or 0
        print(f"turn {turn:>2}: {size:>6} tokens  {'#' * (size // 200)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            break

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments
            print(f"           -> {name}({args})")

            if name in REGISTRY:
                result = REGISTRY[name](**args)
            else:
                result = {"error": f"no tool named '{name}'"}

            # the line that grows the context - results are appended, never removed
            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": json.dumps(result),
            })
    else:
        print("stopped: hit max_turns")

    print(f"messages in history: {len(messages)}")
    return messages

In [27]:
import time
start= time.time()
#history = run("HI")
# history = run("I'm flying from Delhi (DEL) to Goa (GOI) on 2026-12-15, Find me flights, a 4-star hotel in Goa, and a few things to do there.")
history = run("What's the weather like in Manali in December?")
end= time.time()

print(f"Difference:- {end-start}")

turn  1:   2090 tokens  ##########
           -> get_weather({'city': 'Manali', 'month': 12})
turn  2:   2154 tokens  ##########

ANSWER: In December, Manali is **very cold** and is one of the best times to visit if you're looking for snow.

**Weather Summary:**
*   **Temperature:** Highs of around **8°C** and lows dropping to **-2°C** (or lower).
*   **Conditions:** Generally cold with a high chance of **snowfall**, which makes the landscape beautiful but can occasionally impact travel.

**Quick Tips for December:**
*   **Packing:** You'll need heavy woolens—thermal innerwear, a heavy down jacket, waterproof boots, gloves, and a woollen cap.
*   **Travel Alert:** Since snowfall is common, keep a buffer day in your itinerary as roads (especially the Atal Tunnel or roads to Rohtang) can sometimes close due to heavy snow.

messages in history: 5
Difference:- 2.851367950439453


### AGENTIC LOOP WITH DEFERRED TOOL CALLING

In [28]:
def run_with_deferred_loading(question, max_turns=8, verbose=True):
    tools = [SEARCH_TOOL]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx":8000},   # FIX #1 - avoid silent truncation
        )

        if verbose:
            tokens = response.prompt_eval_count or 0
            print(f"turn {turn}: {tokens} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            return messages

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({
                    "role": "tool", "tool_name": "search_tools",
                    "content": json.dumps({"found": found}),
                })
                for tool_name in found:
                    fn = REGISTRY[tool_name]
                    if fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({
                    "role": "tool", "tool_name": name,
                    "content": json.dumps(result),
                })

    print("stopped: hit max_turns without a final answer")
    return messages

In [29]:
import time
start= time.time()
#history = run_with_deferred_loading("HI")
# history = run_with_deferred_loading("I'm flying from Delhi (DEL) to Goa (GOI) on 2026-12-15, Find me flights, a 4-star hotel in Goa, and a few things to do there.")
history = run_with_deferred_loading("What's the weather like in Manali in December?")
end= time.time()

print(f"Difference:- {end-start}")

turn 1: 701 tokens, tools loaded = 1
  [search] {'query': 'weather forecast'} -> ['get_weather']
turn 2: 814 tokens, tools loaded = 2
  [call] get_weather({'city': 'Manali', 'month': 12})
turn 3: 882 tokens, tools loaded = 2

ANSWER: December in Manali is **very cold and typically snowy**, making it a prime time for travelers who want to experience a white winter.

**Weather Breakdown:**
*   **Temperature:** Highs hover around **8°C**, but nighttime temperatures frequently drop to **-2°C** or lower.
*   **Conditions:** It is cold and crisp with a significant chance of snowfall.

**Practical Tips for Indian Travelers:**
1.  **Packing:** This is not the time for a light jacket. You will need heavy woolens, thermal innerwear, a high-quality puffer jacket, waterproof boots (essential for snow), and gloves/beanies.
2.  **Logistics:** Be aware that heavy snowfall can sometimes lead to road blockages on the way to Manali or to nearby spots like Rohtang Pass (which is usually closed in Dec) or

### RAG

In [30]:
### LOADING OF DOCUMENT


import ollama

MODEL = "gpt-oss:120b-cloud"                       
FILE_PATH = "yatra_concierge_knowledge_base.md"

full_doc = open(FILE_PATH, encoding="utf-8").read()
print("document loaded:", len(full_doc), "characters")

document loaded: 23262 characters


In [31]:
## CREATING CHUNKS

parts = full_doc.split("\n## ")
chunks = [parts[0]] + ["## " + p for p in parts[1:]]
chunks = [c.strip() for c in chunks if len(c.strip()) > 100]

print("total chunks:", len(chunks))

total chunks: 21


In [32]:
chunks[2]

'## 2. How Booking Works with Yatra Concierge\n\nWhen you book through Yatra Concierge, you are booking directly with the airline or hotel; we act as your agent and manage the booking on your behalf. Every booking generates a booking reference (for flights, a six-character PNR; for hotels, a confirmation ID) that you can use to view, change, or cancel your trip.\n\nA booking is only confirmed once payment is successful and you receive a confirmation message by email and SMS. If your payment is deducted but you do not receive a confirmation within thirty minutes, do not rebook immediately — contact support first, because a duplicate booking may be created and the second one will follow the standard cancellation rules, which may not be fully refundable.\n\nFor flights, the fare you see is a total fare that already includes base fare, airline fuel charges, and government taxes, unless a separate "add-on" such as extra baggage or seat selection is chosen. For hotels, the price shown is per

In [33]:
## CONETEXTUAL RETRIEVAL 

CONTEXT_PROMPT = """<document>
{document}
</document>
Here is the chunk we want to situate within the whole document
<chunk>
{chunk}
</chunk>
Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else."""



def generate_context(document: str, chunk: str, model: str = MODEL) -> str:
    prompt = CONTEXT_PROMPT.format(document=document, chunk=chunk)
    resp = ollama.generate(model=model, prompt=prompt, options={"temperature": 0.0})
    return resp.response.strip()

In [34]:
!pip install faiss-cpu

In [35]:
## CREATING EMBEDDING HELPER

def embed(text):
    # turn text into a 768-dim vector using nomic-embed-text
    return ollama.embed(model="embeddinggemma:latest", input=text)["embeddings"][0]

In [36]:
import faiss, numpy as np

# ------------------------------------------------------------------
# Build the knowledge base with FAISS:
#   for each chunk -> make context -> glue on top -> embed.
# FAISS stores the VECTORS; a parallel list stores the chunk TEXT
# ------------------------------------------------------------------

vectors = []     # stores the embeddings (numbers)
kb_meta = []     # to store actual text

for i, chunk in enumerate(chunks):   #iterate over each chunk
    context  = generate_context(full_doc, chunk)   # context line for this chunk
    enriched = context + "\n\n" + chunk            # enriched = context + original chunk

    vectors.append(embed(enriched))                # embed the ENRICHED text
    kb_meta.append({                               # but keep the ORIGINAL chunk to return
        "chunk": chunk,
        "section": chunk.splitlines()[0].lstrip("# ").strip(),
    })
    print(f"indexed {i + 1}/{len(chunks)}")

# FAISS needs a float32 numpy array
vectors = np.array(vectors, dtype="float32")  # converting vectors into numeric as fiass accepts vectors in form of numbers

# normalize the vectors so that inner-product search == cosine similarity
faiss.normalize_L2(vectors)

# adding vectors to database
index = faiss.IndexFlatIP(768)
index.add(vectors)

print("knowledge base ready:", index.ntotal, "chunks")

indexed 1/21
indexed 2/21
indexed 3/21
indexed 4/21
indexed 5/21
indexed 6/21
indexed 7/21
indexed 8/21
indexed 9/21
indexed 10/21
indexed 11/21
indexed 12/21
indexed 13/21
indexed 14/21
indexed 15/21
indexed 16/21
indexed 17/21
indexed 18/21
indexed 19/21
indexed 20/21
indexed 21/21
knowledge base ready: 21 chunks


In [37]:
def search_knowledge_base(query: str, k: int = 3) -> list:
    """Search the travel policy & information handbook for rules and fine-print.

    Use this for questions that are NOT live data — cancellation and refund rules,
    baggage allowance, visa and entry requirements, check-in and hotel policies,
    travel insurance, pet travel rules, and seasonal advisories.

    Args:
        query: what to look up, e.g. "hotel cancellation refund policy".
        k: number of passages to return (default 3).
    Returns:
        A list of relevant passages (text) from the handbook.
    """
    # embed the query and normalize it the same way we did the chunks
    q = np.array([embed(query)], dtype="float32")
    faiss.normalize_L2(q)

    # search: returns similarity scores and the row ids of the best matches
    scores, ids = index.search(q, k)

    # map row ids back to the ORIGINAL chunk text
    return [kb_meta[i]["chunk"] for i in ids[0]]



In [38]:
# ------------------------------------------------------------------
# Tell the agent it HAS a knowledge base and when to use it.
# Run this cell ONCE (it appends to SYSTEM_PROMPT).
# ------------------------------------------------------------------
SYSTEM_PROMPT = SYSTEM_PROMPT + """
<knowledge_base>
- Besides live-data tools, you have a policy & information handbook covering rules
  and fine-print: cancellation and refund rules, baggage limits, visa and entry
  requirements, hotel and check-in policies, travel insurance, pet travel, and
  seasonal advisories.
- For any question about these rules or policies, use the `search_knowledge_base`
  tool (find it via `search_tools` if it isn't loaded yet) and answer ONLY from the
  passages it returns.
- If the handbook does not contain the answer, say you don't have that information
  rather than guessing.
</knowledge_base>
"""

In [39]:
print(SYSTEM_PROMPT)


<role>
You are an AI travel concierge for Indian travelers. You help users plan trips
end to end — flights, stays, activities, budgets, and logistics — and you give
clear, decision-ready, honest recommendations.
</role>

<capabilities>
- Planning itineraries and day-by-day schedules
- Comparing flight and hotel options on price, time, and convenience
- Estimating trip budgets and flagging hidden costs
- Practical logistics: visas, weather, local transport, and what to pack
</capabilities>

<behavior>
- If key details are missing, ask for them first: origin city, destination,
  dates (or rough timing), number of travelers, and rough budget.
- Recommend, don't just list. Give specific choices and say why.
- Be honest about trade-offs (e.g. cheaper fare but a longer layover).
- For anything that needs live data — prices, seat availability, weather,
  current visa rules — say so clearly. Never invent prices, flight numbers,
  or policies.
</behavior>

<output>
- Reply in the traveler's la

In [40]:
# register the tool so deferred loading can discover and run it
if search_knowledge_base not in REAL_TOOLS:
    REAL_TOOLS.append(search_knowledge_base)
REGISTRY = {f.__name__: f for f in REAL_TOOLS}

print("tools now available:", list(REGISTRY.keys()))

tools now available: ['search_flights', 'search_hotels', 'get_weather', 'convert_currency', 'get_visa_rules', 'get_flight_status', 'search_activities', 'search_trains', 'search_restaurants', 'estimate_trip_budget', 'get_local_transport', 'get_travel_advisory', 'get_packing_list', 'get_public_holidays', 'get_airport_info', 'book_flight', 'search_knowledge_base']


In [41]:
import numpy as np

# Precompute ONE vector per tool, from its name + first docstring line.
# Run this AFTER search_knowledge_base has been added to REAL_TOOLS.
TOOL_VECS = {
    f.__name__: np.array(embed(f.__name__ + ": " + (f.__doc__ or "").strip().splitlines()[0]))
    for f in REAL_TOOLS
}

def search_tools(query, max_results=3):
    q = np.array(embed(query))
    # cosine similarity between the query and every tool
    scores = {
        name: float(q @ v / (np.linalg.norm(q) * np.linalg.norm(v)))
        for name, v in TOOL_VECS.items()
    }
    # return the most relevant tools first
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:max_results]

In [42]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_tools",
        "description": (
            "Search for tools that are not yet loaded into this conversation. "
            "Right now, this search tool is the only capability you have - "
            "every other tool must be found through this one before it can "
            "be called. Use it the moment you realize you need a capability "
            "you don't currently have.\n\n"
            "Returns a list of matching tool names. Once a tool is returned "
            "here, it becomes directly callable on your next turn - you do "
            "not need to search for it again later in this conversation.\n\n"
            "If nothing matches, try again with broader or different keywords "
            "rather than giving up."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Short keywords describing the capability you need - "
                        "not a full sentence. "
                        "Good: 'search flights', 'hotel cancellation policy', "
                        "'visa rules', 'convert currency'. "
                        "Avoid: a complete question or a full description "
                        "of the task."
                    ),
                },
                "max_results": {
                    "type": "integer",
                    "description": (
                        "Maximum number of matching tools to return. "
                        "Defaults to 3. Increase only if the task clearly "
                        "needs several related tools at once."
                    ),
                    "default": 3,
                },
            },
            "required": ["query"],
        },
    },
}

In [43]:
def run_with_deferred_loading(question, max_turns=8, verbose=True):
    tools = [SEARCH_TOOL]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx":8000},   # FIX #1 - avoid silent truncation
        )

        if verbose:
            tokens = response.prompt_eval_count or 0
            print(f"turn {turn}: {tokens} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            return messages

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({
                    "role": "tool", "tool_name": "search_tools",
                    "content": json.dumps({"found": found}),
                })
                for tool_name in found:
                    fn = REGISTRY[tool_name]
                    if fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({
                    "role": "tool", "tool_name": name,
                    "content": json.dumps(result),
                })

    print("stopped: hit max_turns without a final answer")
    return messages

In [44]:
## TESTING 

run_with_deferred_loading("I'm so excited, this is my first trip abroad!")

turn 1: 847 tokens, tools loaded = 1

ANSWER: That’s wonderful—congratulations on your first overseas adventure! 🎉  

To put together a plan that fits you perfectly, could you tell me a bit more about:

1. **Your departure city in India** (e.g., Delhi, Mumbai, Bangalore, etc.)  
2. **Where you’d like to go** (city or country) – or if you’re still deciding, any regions you’re interested in  
3. **Travel dates** (exact dates or rough month/timeframe)  
4. **Number of travelers** and their ages (e.g., adults, children, seniors)  
5. **Your approximate budget** (overall or per‑person, if you have one in mind)  

Once I have these details, I can suggest flights, accommodation, activities, and a day‑by‑day itinerary that’s right for you. Looking forward to hearing more!



[{'role': 'system',
  'content': "\n<role>\nYou are an AI travel concierge for Indian travelers. You help users plan trips\nend to end — flights, stays, activities, budgets, and logistics — and you give\nclear, decision-ready, honest recommendations.\n</role>\n\n<capabilities>\n- Planning itineraries and day-by-day schedules\n- Comparing flight and hotel options on price, time, and convenience\n- Estimating trip budgets and flagging hidden costs\n- Practical logistics: visas, weather, local transport, and what to pack\n</capabilities>\n\n<behavior>\n- If key details are missing, ask for them first: origin city, destination,\n  dates (or rough timing), number of travelers, and rough budget.\n- Recommend, don't just list. Give specific choices and say why.\n- Be honest about trade-offs (e.g. cheaper fare but a longer layover).\n- For anything that needs live data — prices, seat availability, weather,\n  current visa rules — say so clearly. Never invent prices, flight numbers,\n  or polic

In [45]:
## RAG 
run_with_deferred_loading("How much cabin baggage can I carry on a domestic flight?")

turn 1: 848 tokens, tools loaded = 1
  [search] {'max_results': 5, 'query': 'cabin baggage allowance domestic flight India'} -> ['search_knowledge_base', 'search_flights', 'get_airport_info', 'book_flight', 'get_packing_list']
turn 2: 1325 tokens, tools loaded = 6
  [call] search_knowledge_base({'k': 5, 'query': 'cabin baggage allowance domestic flight India'})
turn 3: 2578 tokens, tools loaded = 6

ANSWER: **Cabin‑baggage allowance on domestic Indian flights**

- **One piece of carry‑on baggage** – maximum weight **7 kg** (≈15 lb).  
- **Dimensions** must fit the overhead bin (generally ≤ 55 cm × 35 cm × 25 cm, but check the specific airline).  
- **One additional personal item** – e.g., a laptop bag, handbag, or small backpack – is also allowed, without a separate weight limit (it just has to be small enough to fit under the seat in front of you).  

If the bag exceeds the 7 kg limit or is oversized, the airline will ask you to check it in at the gate, and gate‑check fees are higher 

[{'role': 'system',
  'content': "\n<role>\nYou are an AI travel concierge for Indian travelers. You help users plan trips\nend to end — flights, stays, activities, budgets, and logistics — and you give\nclear, decision-ready, honest recommendations.\n</role>\n\n<capabilities>\n- Planning itineraries and day-by-day schedules\n- Comparing flight and hotel options on price, time, and convenience\n- Estimating trip budgets and flagging hidden costs\n- Practical logistics: visas, weather, local transport, and what to pack\n</capabilities>\n\n<behavior>\n- If key details are missing, ask for them first: origin city, destination,\n  dates (or rough timing), number of travelers, and rough budget.\n- Recommend, don't just list. Give specific choices and say why.\n- Be honest about trade-offs (e.g. cheaper fare but a longer layover).\n- For anything that needs live data — prices, seat availability, weather,\n  current visa rules — say so clearly. Never invent prices, flight numbers,\n  or polic

In [46]:
run_with_deferred_loading("How many days does a refund take to reach my card?")

turn 1: 848 tokens, tools loaded = 1
  [search] {'max_results': 5, 'query': 'refund card days policy knowledge base'} -> ['search_knowledge_base', 'get_public_holidays', 'search_flights', 'search_trains', 'get_visa_rules']
turn 2: 1329 tokens, tools loaded = 6
  [call] search_knowledge_base({'k': 5, 'query': 'refund to card how many days'})
turn 3: 2707 tokens, tools loaded = 6

ANSWER: **Refund timeline**

According to Yatra Concierge’s policy handbook, refunds are processed back to the original payment method after the airline or hotel releases the money to us. The time you see on your card depends on how you paid:

| Payment method | Typical time for the refund to appear on your card |
|----------------|----------------------------------------------------|
| UPI / digital wallets | **3 – 5 working days** |
| Credit or debit card | **7 – 10 working days** |
| Net‑banking | **7 – 10 working days** |

The clock starts when the airline/hotel issues the refund, not when you request the c

[{'role': 'system',
  'content': "\n<role>\nYou are an AI travel concierge for Indian travelers. You help users plan trips\nend to end — flights, stays, activities, budgets, and logistics — and you give\nclear, decision-ready, honest recommendations.\n</role>\n\n<capabilities>\n- Planning itineraries and day-by-day schedules\n- Comparing flight and hotel options on price, time, and convenience\n- Estimating trip budgets and flagging hidden costs\n- Practical logistics: visas, weather, local transport, and what to pack\n</capabilities>\n\n<behavior>\n- If key details are missing, ask for them first: origin city, destination,\n  dates (or rough timing), number of travelers, and rough budget.\n- Recommend, don't just list. Give specific choices and say why.\n- Be honest about trade-offs (e.g. cheaper fare but a longer layover).\n- For anything that needs live data — prices, seat availability, weather,\n  current visa rules — say so clearly. Never invent prices, flight numbers,\n  or polic

In [47]:
#RAG AND TOOL
run_with_deferred_loading("Find a 4-star hotel in Goa, and what's the cancellation policy if I cancel late?")

turn 1: 855 tokens, tools loaded = 1
  [search] {'max_results': 5, 'query': 'hotel search Goa 4-star'} -> ['search_hotels', 'search_knowledge_base', 'search_flights', 'search_restaurants', 'search_activities']
turn 2: 1364 tokens, tools loaded = 6
  [call] search_hotels({'city': 'Goa', 'max_price_inr': 20000, 'min_stars': 4})
turn 3: 1663 tokens, tools loaded = 6
  [call] search_knowledge_base({'k': 3, 'query': 'hotel cancellation policy late cancellation 4 star hotel India'})
turn 4: 2561 tokens, tools loaded = 6

ANSWER: **Recommended 4‑star hotel in Goa**

| Hotel | Area | Nightly rate (INR) | Rating (out of 5) | Breakfast |
|-------|------|-------------------|-------------------|-----------|
| **Beira Mar Resort** | Baga | 5,200 | 4.3 | Yes |
| **Palm Grove Boutique** | Anjuna | 6,100 | 4.5 | Yes |

**Why I recommend Palm Grove Boutique**

- **Higher guest rating (4.5)** – consistently praised for clean rooms, friendly staff and a great location close to Anjuna Beach and the famous

[{'role': 'system',
  'content': "\n<role>\nYou are an AI travel concierge for Indian travelers. You help users plan trips\nend to end — flights, stays, activities, budgets, and logistics — and you give\nclear, decision-ready, honest recommendations.\n</role>\n\n<capabilities>\n- Planning itineraries and day-by-day schedules\n- Comparing flight and hotel options on price, time, and convenience\n- Estimating trip budgets and flagging hidden costs\n- Practical logistics: visas, weather, local transport, and what to pack\n</capabilities>\n\n<behavior>\n- If key details are missing, ask for them first: origin city, destination,\n  dates (or rough timing), number of travelers, and rough budget.\n- Recommend, don't just list. Give specific choices and say why.\n- Be honest about trade-offs (e.g. cheaper fare but a longer layover).\n- For anything that needs live data — prices, seat availability, weather,\n  current visa rules — say so clearly. Never invent prices, flight numbers,\n  or polic

In [48]:
run_with_deferred_loading("Book me a Delhi to Goa flight on 2026-12-15 and tell me the cabin baggage limit.")

turn 1: 859 tokens, tools loaded = 1
  [search] {'max_results': 5, 'query': 'flight search'} -> ['search_flights', 'get_flight_status', 'search_knowledge_base', 'search_trains', 'get_airport_info']
turn 2: 1343 tokens, tools loaded = 6
  [call] search_flights({'date': '2026-12-15', 'origin_code': 'DEL', 'destination_code': 'GOI'})
turn 3: 1948 tokens, tools loaded = 6
  [call] search_knowledge_base({'k': 5, 'query': 'cabin baggage limit IndiGo'})
turn 4: 3184 tokens, tools loaded = 6

ANSWER: **Your Delhi → Goa flight (15 Dec 2026)**  

| Option | Airline | Dep‑time | Arr‑time | Duration | Stops | Price (INR) | Seats left |
|--------|---------|----------|----------|----------|------|--------------|------------|
| **1** (Best value) | IndiGo (6E‑231) | 06:10 am | 08:45 am | 2 h 35 m | Non‑stop | **5,499** | 7 |
| **2** | SpiceJet (SG‑133) | 14:05 pm | 16:40 pm | 2 h 35 m | Non‑stop | 5,299 | 9 |
| **3** | IndiGo (6E‑577) | 11:20 am | 13:55 pm | 2 h 35 m | Non‑stop | 5,899 | 12 |
| 4 | A

[{'role': 'system',
  'content': "\n<role>\nYou are an AI travel concierge for Indian travelers. You help users plan trips\nend to end — flights, stays, activities, budgets, and logistics — and you give\nclear, decision-ready, honest recommendations.\n</role>\n\n<capabilities>\n- Planning itineraries and day-by-day schedules\n- Comparing flight and hotel options on price, time, and convenience\n- Estimating trip budgets and flagging hidden costs\n- Practical logistics: visas, weather, local transport, and what to pack\n</capabilities>\n\n<behavior>\n- If key details are missing, ask for them first: origin city, destination,\n  dates (or rough timing), number of travelers, and rough budget.\n- Recommend, don't just list. Give specific choices and say why.\n- Be honest about trade-offs (e.g. cheaper fare but a longer layover).\n- For anything that needs live data — prices, seat availability, weather,\n  current visa rules — say so clearly. Never invent prices, flight numbers,\n  or polic

### MEMORY

In [49]:
# ------------------------------------------------------------------
# SHORT-TERM MEMORY lives OUTSIDE the function, so it survives across
# turns. Re-run THIS cell to start a fresh conversation.
# ------------------------------------------------------------------
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
tools    = [SEARCH_TOOL]        # loaded tools also persist across turns


def agent_turn(question, max_turns=8, verbose=True):
    # add the new user message to the SAME growing history
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx": 8000},
        )

        if verbose:
            print(f"  turn {turn}: {response.prompt_eval_count or 0} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        # no tool call -> this is the final answer for this question
        if not response.message.tool_calls:
            return response.message.content

        for call in response.message.tool_calls:
            name, args = call.function.name, call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({"role": "tool", "tool_name": "search_tools",
                                 "content": json.dumps({"found": found})})
                for tool_name in found:                 # load the discovered tools
                    fn = REGISTRY.get(tool_name)
                    if fn and fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({"role": "tool", "tool_name": name,
                                 "content": json.dumps(result)})

    return "(stopped: hit max_turns without a final answer)"


# ------------------------------------------------------------------
# CHAT LOOP: keeps talking until the user types 'exit' or 'quit'
# ------------------------------------------------------------------
def chat():
    print("🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.\n")
    while True:
        question = input("You: ").strip()

        if question.lower() in ("exit", "quit"):
            print("Concierge: Safe travels! 👋")
            break
        if not question:
            continue

        answer = agent_turn(question)
        print(f"\nConcierge: {answer}\n")

In [50]:
chat()

🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.



You:  exit


Concierge: Safe travels! 👋


In [51]:
turns = [
    "Hi! I'm Ajay, based in Delhi. I'm planning a trip to Goa next month.",
    "Find me flights from Delhi (DEL) to Goa (GOI) on 2026-12-20.",
    "i will choose :- Vistara UK‑849, By the way I always fly window seat. Now find me a boutique hotel in Goa — I like to keep it under 6000 a night.",
    "I'm also bringing my dog along. What do I need to arrange for that?",
    "Heads up, I'm allergic to peanuts — keep that in mind for food. Any good vegetarian restaurants in Goa?",
    "And what's the hotel cancellation policy if I have to cancel late?",
]



In [52]:
messages

[{'role': 'system',
  'content': "\n<role>\nYou are an AI travel concierge for Indian travelers. You help users plan trips\nend to end — flights, stays, activities, budgets, and logistics — and you give\nclear, decision-ready, honest recommendations.\n</role>\n\n<capabilities>\n- Planning itineraries and day-by-day schedules\n- Comparing flight and hotel options on price, time, and convenience\n- Estimating trip budgets and flagging hidden costs\n- Practical logistics: visas, weather, local transport, and what to pack\n</capabilities>\n\n<behavior>\n- If key details are missing, ask for them first: origin city, destination,\n  dates (or rough timing), number of travelers, and rough budget.\n- Recommend, don't just list. Give specific choices and say why.\n- Be honest about trade-offs (e.g. cheaper fare but a longer layover).\n- For anything that needs live data — prices, seat availability, weather,\n  current visa rules — say so clearly. Never invent prices, flight numbers,\n  or polic

### LTM

In [49]:
!pip install -U mem0ai

In [50]:
from mem0 import Memory

In [51]:
CUSTOM_INSTRUCTIONS = """
You are extracting long-term memory for an AI Travel Concierge.
Only store facts that are STABLE and USEFUL for planning this traveller's FUTURE trips.
A memory must still be true and relevant weeks or months from now. If a detail is
tied to one specific trip or one moment, it is NOT a memory.

STORE (these count as memory):
- Identity & home base: name, home city, home airport, nationality (for visa needs).
- Travel preferences: preferred airlines, seat choice (window/aisle), cabin class,
  nonstop vs cheapest, preferred hotel type (boutique, luxury, budget), and any
  usual budget range or per-night limit.
- Constraints & needs: dietary restrictions and food allergies, accessibility or
  mobility needs, travelling with a pet, travelling with kids or elderly family.
- Loyalty & documents held: frequent-flyer / hotel loyalty programs, whether they
  hold a valid passport or specific visas (store the FACT, never the number).
- Stable likes/dislikes: types of destinations or activities they consistently
  prefer or avoid (e.g. "prefers beaches over trekking", "dislikes long layovers").
- Recurring patterns: destinations they visit often, who they usually travel with.

DO NOT STORE (these are NOT memory):
- Small talk, greetings, moods, or comments (traffic, weather complaints, "coffee went cold").
- One-off trip details unless stated as a lasting preference (a single date, a single
  destination they asked about once, a specific booking for one trip).
- Live/temporary data pulled from tools: flight numbers, prices, seat availability,
  today's weather, currency rates, PNRs, booking confirmations.
- Arithmetic, quick lookups, or anything the traveller asked just to check something.
- Sensitive identifiers: passport numbers, card or bank numbers, government IDs.

RULES:
- Write each memory as one short, self-contained fact in simple language
  (e.g. "Is allergic to peanuts", "Prefers boutique hotels under 6000 INR/night").
- Prefer the traveller's own stated words; never guess or infer beyond what they said.
- If nothing durable was said, extract nothing.
"""

In [52]:
config = {
    "llm":      {"provider": "ollama", "config": {"model": "gemma4:31b-cloud"}},
    "embedder": {"provider": "ollama", "config": {"model": "embeddinggemma:latest"}},
    "vector_store": {
        "provider": "qdrant",
        "config": {"embedding_model_dims": 768, "collection_name": "mem0_gemma3"},
    },
    "custom_instructions": CUSTOM_INSTRUCTIONS,
    "version": "v1.1",
}

m = Memory.from_config(config)
print("Mem0 ready.")

[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


Mem0 ready.


In [53]:
m.delete_all(user_id="ajay")
print("Cleared Ajay's memories.")

Cleared Ajay's memories.


In [58]:
result1= m.add(messages, user_id="ajay")   # extracts durable facts, ignores the noise
print("Memory created.")

E:\Anaconda\Lib\site-packages\spacy\cli\_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
E:\Anaconda\Lib\site-packages\weasel\util\config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"


Memory created.


E:\Anaconda\Lib\site-packages\mem0\vector_stores\qdrant.py:177: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(


In [59]:
# SHOWING MEMORY 
for mem in m.get_all(filters={"user_id": "ajay"})["results"]:
    print("-", mem["memory"])

- User's name is Ajay and he is based in Delhi


In [60]:
#HOW ITEMS ARE ACTUALLY STORED IN MEMORY 
for item in result1["results"]:
    print(m.get(item["id"]))
    print("**********")

{'id': '0b18214d-eb4f-4da7-83e7-f5fe56352b1f', 'memory': "User's name is Ajay and he is based in Delhi", 'hash': '91b66067f8a87e914925b21cfb9b4fae', 'metadata': None, 'score': None, 'created_at': '2026-09-22T04:41:16.965467+00:00', 'updated_at': '2026-09-22T04:41:16.965467+00:00', 'user_id': 'ajay', 'attributed_to': 'user'}
**********


In [61]:
m.search("Which food is user intolerant too?",
         filters={"user_id":"ajay"})

{'results': [{'id': '0b18214d-eb4f-4da7-83e7-f5fe56352b1f',
   'memory': "User's name is Ajay and he is based in Delhi",
   'hash': '91b66067f8a87e914925b21cfb9b4fae',
   'metadata': None,
   'score': 0.3100561062128889,
   'created_at': '2026-09-22T04:41:16.965467+00:00',
   'updated_at': '2026-09-22T04:41:16.965467+00:00',
   'user_id': 'ajay',
   'attributed_to': 'user'}]}

In [62]:
m.search("Where do i live ??",
         filters={"user_id":"ajay"})

{'results': [{'id': '0b18214d-eb4f-4da7-83e7-f5fe56352b1f',
   'memory': "User's name is Ajay and he is based in Delhi",
   'hash': '91b66067f8a87e914925b21cfb9b4fae',
   'metadata': None,
   'score': 0.5273270450213232,
   'created_at': '2026-09-22T04:41:16.965467+00:00',
   'updated_at': '2026-09-22T04:41:16.965467+00:00',
   'user_id': 'ajay',
   'attributed_to': 'user'}]}

In [60]:
def get_top_memories(query, user_id="ajay", top_k=2):
    response = m.search(
        query,
        filters={"user_id": user_id}
    )

    top_memories = [
        result["memory"]
        for result in response["results"][:top_k]
    ]

    return top_memories


In [64]:
# Example
memories = get_top_memories("Where do i dont prefer to eat?")
print(memories)

["User's name is Ajay and he is based in Delhi"]


In [65]:
user_question= "Where do i dont prefer to eat?"

In [66]:
user_question= f"<user_question>{user_question}</user_question>  <relevant_user_info>{memories} </relevant_user_info>"

In [67]:
user_question

'<user_question>Where do i dont prefer to eat?</user_question>  <relevant_user_info>["User\'s name is Ajay and he is based in Delhi"] </relevant_user_info>'

In [68]:
# ------------------------------------------------------------------
# SHORT-TERM MEMORY lives OUTSIDE the function, so it survives across
# turns. Re-run THIS cell to start a fresh conversation.
# ------------------------------------------------------------------
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
tools    = [SEARCH_TOOL]        # loaded tools also persist across turns


def agent_turn(question, max_turns=8, verbose=True):
    memories = get_top_memories(question)
    question= f"<user_question>{question}</user_question>  <user_relevant_info>{memories}</user_relevant_info>"
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx": 8000},
        )

        if verbose:
            print(f"  turn {turn}: {response.prompt_eval_count or 0} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        # no tool call -> this is the final answer for this question
        if not response.message.tool_calls:
            return response.message.content

        for call in response.message.tool_calls:
            name, args = call.function.name, call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({"role": "tool", "tool_name": "search_tools",
                                 "content": json.dumps({"found": found})})
                for tool_name in found:                 # load the discovered tools
                    fn = REGISTRY.get(tool_name)
                    if fn and fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({"role": "tool", "tool_name": name,
                                 "content": json.dumps(result)})

    return "(stopped: hit max_turns without a final answer)"


# ------------------------------------------------------------------
# CHAT LOOP: keeps talking until the user types 'exit' or 'quit'
# ------------------------------------------------------------------
def chat():
    print("🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.\n")
    while True:
        question = input("You: ").strip()

        if question.lower() in ("exit", "quit"):
            print("Concierge: Safe travels! 👋")
            break
        if not question:
            continue

        answer = agent_turn(question)
        print(f"\nConcierge: {answer}\n")

In [69]:
chat()

🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.



You:  exot


  turn 1: 873 tokens, tools loaded = 1

Concierge: Hi Ajay!  

I see you’re interested in something exotic, but I need a few more details to craft a perfect plan for you. Could you let me know:

1. **Who’s traveling?** (just you, family, friends, kids, etc.)  
2. **When do you want to go?** (exact dates or a general month/season)  
3. **What’s your approximate budget (per person) in INR?**  
4. **Any specific interests?** (beaches, wildlife, culture, adventure, food, etc.)  
5. **Do you have any preferred departure city?** (I’ll assume Delhi unless you say otherwise)

Once I have this info, I can suggest exotic destinations, flight options, hotels, activities, and a rough cost estimate. Looking forward to your reply!



You:  exit


Concierge: Safe travels! 👋


In [70]:
turns = [
    "Hi! I'm Ajay, based in Delhi. I'm planning a trip to Goa next month and i prefer response in english.",
    "Find me flights from Delhi (DEL) to Goa (GOI) on 2026-12-20.",
    "i will choose :- Vistara UK‑849, By the way I always fly window seat. Now find me a boutique hotel in Goa — I like to keep it under 6000 a night.",
    "I'm also bringing my dog along. What do I need to arrange for that?",
    "Heads up, I'm allergic to peanuts — keep that in mind for food. Any good vegetarian restaurants in Goa?",
    "And what's the hotel cancellation policy if I have to cancel late?",
    " -------" 
]



In [71]:
#### SUMMARIZATION

In [54]:
SUMMARY_SYSTEM = """You are a conversation summarizer. You keep ONE short, up-to-date summary so the conversation can continue without the full history.

You are given the existing summary (may be empty) and new turns to fold in. Produce a SINGLE merged summary. Integrate the new info and drop anything
now outdated.

Output these headings, each followed by ONE line. If a heading has nothing, write "None yet.":

GOALS:
DECISIONS:
FACTS:
PROGRESS:
OPEN:
NEXT:

STYLE — compact but complete:
- Telegraphic. Fragments, not sentences. Separate items with semicolons.
- Keep EVERY decision, fact, open question — lose nothing. Only cut filler words, reasons, and repetition, never information.
- Copy exact values verbatim (dates, numbers, budgets, names). These are the priority.
- No explanations, no reasons "why", no pleasantries, no narrative.
- Whole summary under 120 words. Compress older items harder than recent ones.
- Do NOT invent anything not actually said. Unknown -> put it under OPEN.
- If new info contradicts the old summary, keep the newer, drop the old.

Target format (style, not content):
GOALS: 7-day Japan trip; April; 2 people; from Delhi.
DECISIONS: Tokyo+Kyoto only; skip JR Pass; Shinkansen point-to-point; IC cards.
FACTS: budget ₹2L (flights ₹1.3L, train ₹32k, hotel+food ₹38k); vegetarian; likes temples/gardens/nature, not nightlife.
PROGRESS: route, transport, visa steps, packing, day-by-day outline done.
OPEN: passports valid 6+ months?; specific hotel names?; booking timeline?
NEXT: build visa-ready itinerary table; pick hotels."""


In [73]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
tools    = [SEARCH_TOOL]        # loaded tools also persist across turns
TRIGGER =5000

def agent_turn(question, max_turns=8, verbose=True):
    global messages
    memories = get_top_memories(question)
    question= f"<user_question>{question}</user_question>  <user_relevant_info>{memories}</user_relevant_info>"
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx": 8000},
        )

        if verbose:
            print(f"  turn {turn}: {response.prompt_eval_count or 0} tokens, tools loaded = {len(tools)}")

        tokens = response.prompt_eval_count or 0
        if tokens > TRIGGER:
            print(f"[auto-compact: last prompt was {tokens} tokens, "
                  f"threshold is {TRIGGER}]")

            system_msg = messages[0]
            old, recent = messages[1:-1], messages[-1:]

            old_text = "\n".join(
                f"[{m['role']}] {m['content']}"
                for m in old
            )

            summary = ollama.chat(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SUMMARY_SYSTEM},
                    {"role": "user", "content": old_text}
                ]
            )

            note = {
                "role": "user",
                "content": f"[conversation so far]: "
                           f"{summary['message']['content']}"
            }

            messages = [system_msg, note] + recent

            print(f"[compacted: {len(old)} messages replaced by 1 summary - "
                  f"watch the next turn's token count drop]")

        messages.append(response.message)

        # no tool call -> this is the final answer for this question
        if not response.message.tool_calls:
            return response.message.content

        for call in response.message.tool_calls:
            name, args = call.function.name, call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({"role": "tool", "tool_name": "search_tools",
                                 "content": json.dumps({"found": found})})
                for tool_name in found:                 # load the discovered tools
                    fn = REGISTRY.get(tool_name)
                    if fn and fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({"role": "tool", "tool_name": name,
                                 "content": json.dumps(result)})

    return "(stopped: hit max_turns without a final answer)"

In [74]:
chat()

🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.



You:  exit


Concierge: Safe travels! 👋


In [75]:
turns = [
    "Hi! I'm Ajay, based in Delhi. I'm planning a trip to Goa next month and i prefer response in english.",
    "Find me flights from Delhi (DEL) to Goa (GOI) on 2026-12-20.",
    "i will choose :- Vistara UK‑849, By the way I always fly window seat. Now find me a boutique hotel in Goa — I like to keep it under 6000 a night.",
    "I'm also bringing my dog along. What do I need to arrange for that?",
    "Heads up, I'm allergic to peanuts — keep that in mind for food. Any good vegetarian restaurants in Goa?",
    "And what's the hotel cancellation policy if I have to cancel late?",
]



In [66]:
# SUMMARY AND MEMORY TOGETHER
def create_summary(conversation):
    # JUST creates a summary: fold these turns into one compact summary.
    old_text = "\n".join(
        f"[{msg['role']}] {msg['content']}"
        for msg in conversation
    )
    resp = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM},
            {"role": "user",   "content": old_text},
        ],
    )
    return resp["message"]["content"]

USER_ID     = "ajay"
def create_memory(conversation, user_id=USER_ID):
    # JUST creates memory: hand the turns to Mem0, which extracts and
    # classifies durable facts ([SEMANTIC] / [EPISODIC] / [PROCEDURAL]).
    return m.add(conversation, user_id=user_id)

In [80]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
tools    = [SEARCH_TOOL]        # loaded tools also persist across turns
TRIGGER =5000

def agent_turn(question, max_turns=8, verbose=True):
    global messages
    memories = get_top_memories(question)
    question= f"<user_question>{question}</user_question>  <user_relevant_info>{memories}</user_relevant_info>"
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx": 8000},
        )

        if verbose:
            print(f"  turn {turn}: {response.prompt_eval_count or 0} tokens, tools loaded = {len(tools)}")

        tokens = response.prompt_eval_count or 0
        if tokens > TRIGGER:
            print(f"[TRIGGER: {tokens} > {TRIGGER} -> create summary + create memory]")

            system_msg = messages[0]
            old        = messages[1:-1]
            recent     = messages[-1:]

            # memory reads only REAL turns, never a previous summary note
            raw_old = [msg for msg in old
                       if not msg["content"].startswith("[conversation so far]:")]

            summary = create_summary(old)        # <-- function 1
            create_memory(raw_old, USER_ID)      # <-- function 2

            note = {"role": "user", "content": f"[conversation so far]: {summary}"}
            messages = [system_msg, note] + recent

        messages.append(response.message)

        # no tool call -> this is the final answer for this question
        if not response.message.tool_calls:
            return response.message.content

        for call in response.message.tool_calls:
            name, args = call.function.name, call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({"role": "tool", "tool_name": "search_tools",
                                 "content": json.dumps({"found": found})})
                for tool_name in found:                 # load the discovered tools
                    fn = REGISTRY.get(tool_name)
                    if fn and fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({"role": "tool", "tool_name": name,
                                 "content": json.dumps(result)})

    return "(stopped: hit max_turns without a final answer)"

In [81]:
# ------------------------------------------------------------------
# CHAT LOOP: keeps talking until the user types 'exit' or 'quit'
# ------------------------------------------------------------------
def chat():
    print("🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.\n")
    while True:
        question = input("You: ").strip()

        if question.lower() in ("exit", "quit"):
            print("Concierge: Safe travels! 👋")
            break
        if not question:
            continue

        answer = agent_turn(question)
        print(f"\nConcierge: {answer}\n")

In [82]:
chat()

🧳 Travel Concierge ready. Type 'exit' or 'quit' to stop.



You:  which language do i prefer?


  turn 1: 875 tokens, tools loaded = 1

Concierge: You prefer receiving responses in **English**.



You:  exit


Concierge: Safe travels! 👋


In [ ]:
turns = [
    "Hi! I'm Ajay, based in Delhi. I'm planning a trip to Goa next month and i prefer response in english.",
    "Find me flights from Delhi (DEL) to Goa (GOI) on 2026-12-20.",
    "i will choose :- Vistara UK‑849, By the way I always fly window seat. Now find me a boutique hotel in Goa — I like to keep it under 6000 a night.",
    "I'm also bringing my dog along. What do I need to arrange for that?",
    "Heads up, I'm allergic to peanuts — keep that in mind for food. Any good vegetarian restaurants in Goa?",
    "And what's the hotel cancellation policy if I have to cancel late?",
]
